# 02 - Modelagem: MLP PyTorch vs. Baselines

Notebook da Etapa 2: construção e treinamento do MLP com PyTorch, comparação com baselines (lineares + árvores) e registro de todos os experimentos no MLflow.

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score, precision_score,
    recall_score, average_precision_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay, RocCurveDisplay,
    PrecisionRecallDisplay
)
from torch.utils.data import DataLoader
import mlflow

from src.data_loader import load_telco_churn
from src.model import ChurnMLP
from src.training import ChurnDataset, train_model, predict_proba

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

sns.set_theme(style="whitegrid")
%matplotlib inline

## 1. Preparação dos Dados

Carregamento, limpeza e pré-processamento — mesma pipeline da Etapa 1.

In [ ]:
df = load_telco_churn()

df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df.dropna(inplace=True)
df.reset_index(drop=True, inplace=True)
df.drop(columns=['customerID'], inplace=True)

num_cols = df.select_dtypes(include=np.number).columns.tolist()
cat_cols = df.select_dtypes(include='object').columns.tolist()
cat_cols.remove('Churn')

X = df.drop(columns=['Churn'])
y = (df['Churn'] == 'Yes').astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(drop='if_binary', handle_unknown='ignore', sparse_output=False), cat_cols),
    ]
)

print(f"Treino: {X_train.shape[0]} | Teste: {X_test.shape[0]}")
print(f"Proporção Churn — treino: {y_train.mean():.2%} | teste: {y_test.mean():.2%}")

In [ ]:
mlflow.set_experiment("churn-prediction")
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

def evaluate_model(y_true, y_pred, y_proba):
    """Calcula métricas de avaliação."""
    return {
        'accuracy': accuracy_score(y_true, y_pred),
        'f1_score': f1_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall': recall_score(y_true, y_pred),
        'auc_roc': roc_auc_score(y_true, y_proba),
        'pr_auc': average_precision_score(y_true, y_proba),
    }

def log_sklearn_model(pipe, name, params, tag="baseline"):
    """Treina um modelo sklearn, faz CV e loga no MLflow."""
    with mlflow.start_run(run_name=name):
        mlflow.set_tag("stage", tag)
        for k, v in params.items():
            mlflow.log_param(k, v)

        cv_results = cross_validate(
            pipe, X_train, y_train, cv=cv,
            scoring=['accuracy', 'f1', 'roc_auc'],
            return_train_score=False
        )
        for metric in ['accuracy', 'f1', 'roc_auc']:
            scores = cv_results[f'test_{metric}']
            mlflow.log_metric(f"cv_{metric}_mean", scores.mean())
            mlflow.log_metric(f"cv_{metric}_std", scores.std())

        pipe.fit(X_train, y_train)
        y_pred = pipe.predict(X_test)
        y_proba = pipe.predict_proba(X_test)[:, 1]
        metrics = evaluate_model(y_test, y_pred, y_proba)

        for k, v in metrics.items():
            mlflow.log_metric(k, v)

    return metrics, y_pred, y_proba

## 2. Baselines — Lineares e Árvores

Treinamos 4 baselines para estabelecer pisos de comparação com o MLP:
1. **DummyClassifier** — lower bound (prediz classe majoritária)
2. **Regressão Logística** — baseline linear
3. **Random Forest** — ensemble de árvores (bagging)
4. **Gradient Boosting** — ensemble de árvores (boosting)

In [ ]:
results = {}

# DummyClassifier
dummy_pipe = Pipeline([('pre', preprocessor), ('clf', DummyClassifier(strategy='most_frequent', random_state=SEED))])
results['DummyClassifier'], dummy_pred, dummy_proba = log_sklearn_model(
    dummy_pipe, "dummy-classifier", {"model": "DummyClassifier", "strategy": "most_frequent"}
)

# Regressão Logística
lr_pipe = Pipeline([('pre', preprocessor), ('clf', LogisticRegression(max_iter=1000, random_state=SEED, class_weight='balanced'))])
results['Logistic Regression'], lr_pred, lr_proba = log_sklearn_model(
    lr_pipe, "logistic-regression", {"model": "LogisticRegression", "max_iter": 1000, "class_weight": "balanced"}
)

# Random Forest
rf_pipe = Pipeline([('pre', preprocessor), ('clf', RandomForestClassifier(n_estimators=200, random_state=SEED, class_weight='balanced'))])
results['Random Forest'], rf_pred, rf_proba = log_sklearn_model(
    rf_pipe, "random-forest", {"model": "RandomForest", "n_estimators": 200, "class_weight": "balanced"}
)

# Gradient Boosting
gb_pipe = Pipeline([('pre', preprocessor), ('clf', GradientBoostingClassifier(n_estimators=200, random_state=SEED))])
results['Gradient Boosting'], gb_pred, gb_proba = log_sklearn_model(
    gb_pipe, "gradient-boosting", {"model": "GradientBoosting", "n_estimators": 200}
)

print("Baselines treinados e registrados no MLflow.")

---

## 3. MLP com PyTorch

Rede neural Multi-Layer Perceptron:
- **Arquitetura:** Input → 64 (ReLU, BN, Dropout) → 32 (ReLU, BN, Dropout) → 1
- **Loss:** `BCEWithLogitsLoss` com `pos_weight` para compensar desbalanceamento
- **Optimizer:** Adam (lr=1e-3)
- **Early stopping:** patience=15, monitorando validation loss
- **Batch size:** 64

In [ ]:
# Pré-processamento para PyTorch (sem pipeline, precisa dos arrays diretamente)
preprocessor.fit(X_train)
X_train_processed = preprocessor.transform(X_train)
X_test_processed = preprocessor.transform(X_test)

# Split treino → treino + validação (para early stopping)
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train_processed, y_train.values, test_size=0.15, random_state=SEED, stratify=y_train
)

# DataLoaders
train_ds = ChurnDataset(X_tr, y_tr)
val_ds = ChurnDataset(X_val, y_val)
test_ds = ChurnDataset(X_test_processed, y_test.values)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=128, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=128, shuffle=False)

# pos_weight para compensar desbalanceamento
pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

input_dim = X_train_processed.shape[1]
print(f"Features após pré-processamento: {input_dim}")
print(f"pos_weight: {pos_weight:.2f}")

In [ ]:
mlp_params = {
    "model": "ChurnMLP",
    "hidden_dims": [64, 32],
    "dropout": 0.3,
    "lr": 1e-3,
    "batch_size": 64,
    "epochs": 100,
    "patience": 15,
    "pos_weight": round(pos_weight, 2),
    "optimizer": "Adam",
    "loss": "BCEWithLogitsLoss",
}

model = ChurnMLP(input_dim=input_dim, hidden_dims=[64, 32], dropout=0.3)

with mlflow.start_run(run_name="mlp-pytorch"):
    mlflow.set_tag("stage", "neural_network")
    for k, v in mlp_params.items():
        mlflow.log_param(k, v)

    history = train_model(
        model, train_loader, val_loader,
        epochs=100, lr=1e-3, pos_weight=pos_weight,
        patience=15
    )

    # Avaliação no conjunto de teste
    mlp_proba = predict_proba(model, X_test_processed)
    mlp_pred = (mlp_proba >= 0.5).astype(int)
    mlp_metrics = evaluate_model(y_test, mlp_pred, mlp_proba)

    for k, v in mlp_metrics.items():
        mlflow.log_metric(k, v)

    results['MLP (PyTorch)'] = mlp_metrics

    print("\nMLP — Métricas no conjunto de teste:")
    for k, v in mlp_metrics.items():
        print(f"  {k}: {v:.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(history['train_loss'], label='Train Loss')
ax.plot(history['val_loss'], label='Validation Loss')
ax.set_xlabel('Época')
ax.set_ylabel('Loss (BCE)')
ax.set_title('Curva de Treinamento — MLP')
ax.legend()
plt.tight_layout()
plt.show()

---

## 4. Comparação de Modelos

Tabela comparativa com todas as 6 métricas para os 5 modelos treinados.

In [ ]:
comparison = pd.DataFrame(results).T.sort_values('auc_roc', ascending=False)
comparison.style.format('{:.4f}').highlight_max(axis=0, color='lightgreen').highlight_min(axis=0, color='#ffcccc')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

metrics_to_plot = comparison.drop(columns=['accuracy']).drop(index='DummyClassifier')

# Gráfico de barras comparativo
metrics_to_plot.plot(kind='bar', ax=axes[0], edgecolor='black')
axes[0].set_title('Comparação de Métricas por Modelo')
axes[0].set_ylabel('Score')
axes[0].set_ylim(0, 1)
axes[0].legend(loc='lower right', fontsize=8)
axes[0].tick_params(axis='x', rotation=30)

# Curvas ROC
model_probas = {
    'Logistic Regression': lr_proba,
    'Random Forest': rf_proba,
    'Gradient Boosting': gb_proba,
    'MLP (PyTorch)': mlp_proba,
}
for name, proba in model_probas.items():
    RocCurveDisplay.from_predictions(y_test, proba, ax=axes[1], name=name)
axes[1].plot([0, 1], [0, 1], 'k--', label='Random')
axes[1].set_title('Curvas ROC — Todos os Modelos')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

## 5. Análise de Custo — Trade-off FP vs. FN

Comparação do impacto financeiro de cada modelo, considerando:
- **Falso Negativo (FN):** R\$ 500 — cliente cancela sem ser identificado (perda de receita)
- **Falso Positivo (FP):** R\$ 50 — oferta de retenção desnecessária

In [ ]:
CUSTO_FN = 500
CUSTO_FP = 50

all_preds = {
    'DummyClassifier': dummy_pred,
    'Logistic Regression': lr_pred,
    'Random Forest': rf_pred,
    'Gradient Boosting': gb_pred,
    'MLP (PyTorch)': mlp_pred,
}

custo_sem_modelo = y_test.sum() * CUSTO_FN
cost_rows = []

for name, preds in all_preds.items():
    cm = confusion_matrix(y_test, preds)
    tn, fp, fn, tp = cm.ravel()
    custo = fn * CUSTO_FN + fp * CUSTO_FP
    economia = custo_sem_modelo - custo
    cost_rows.append({
        'Modelo': name, 'TP': tp, 'FP': fp, 'FN': fn, 'TN': tn,
        'Custo Total (R$)': custo, 'Economia (R$)': economia
    })

cost_df = pd.DataFrame(cost_rows).set_index('Modelo')
cost_df.style.format({'Custo Total (R$)': '{:,.0f}', 'Economia (R$)': '{:,.0f}'}).highlight_max(
    subset=['Economia (R$)'], color='lightgreen'
)

## 6. Matrizes de Confusão e Classification Report — MLP

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

ConfusionMatrixDisplay.from_predictions(y_test, mlp_pred, ax=axes[0], cmap='Blues')
axes[0].set_title('Matriz de Confusão — MLP')

PrecisionRecallDisplay.from_predictions(y_test, mlp_proba, ax=axes[1], name='MLP')
PrecisionRecallDisplay.from_predictions(y_test, gb_proba, ax=axes[1], name='Gradient Boosting')
axes[1].set_title('Curvas Precision-Recall')

plt.tight_layout()
plt.show()

In [ ]:
print("Classification Report — MLP (PyTorch):\n")
print(classification_report(y_test, mlp_pred, target_names=['No Churn', 'Churn']))

## 7. Salvamento do Modelo MLP

Salva os artefatos necessários para servir o modelo na API (Etapa 3):
- Pesos do modelo (`.pt`)
- Preprocessor ajustado (`.joblib`)
- Metadados de treinamento

In [ ]:
import joblib
from pathlib import Path

models_dir = Path('../models')
models_dir.mkdir(exist_ok=True)

torch.save({
    'model_state_dict': model.state_dict(),
    'input_dim': input_dim,
    'hidden_dims': [64, 32],
    'dropout': 0.3,
    'metrics': mlp_metrics,
}, models_dir / 'mlp_churn.pt')

joblib.dump(preprocessor, models_dir / 'preprocessor.joblib')

print(f"Modelo salvo em: {models_dir / 'mlp_churn.pt'}")
print(f"Preprocessor salvo em: {models_dir / 'preprocessor.joblib'}")